# Семинар 8 [17-25-2]. Случайные величины: функция распределения и плотность

В этом ноутбуке:

1. дискретная случайная величина: ряд распределения, многоугольник и «лесенка» $F(x)$;
2. проверка задач S08-01 – S08-03 в коде;
3. непрерывная случайная величина: плотность $f(x)$, функция $F(x)$ и вероятность как площадь;
4. проверка задач S08-04 – S08-06;
5. гид по `scipy.stats`: какой метод что считает;
6. из данных в распределение: эмпирическая функция распределения и гистограмма;
7. оцениваем параметр распределения по данным;
8. как компьютер генерирует случайную величину с нужной $F(x)$;
9. практика: засуха, обеспеченность осадков и кривая $1-F(x)$.

Запускайте ячейки по порядку: **Среда выполнения → Выполнить все** или `Shift+Enter` по одной.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate

rng = np.random.default_rng(2026)   # фиксируем зерно, чтобы у всех получались одинаковые картинки
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Дискретная случайная величина

**Ряд распределения** — таблица «значение → вероятность». Сумма вероятностей равна 1.

**Функция распределения** на лекции определена как $F(x)=P(X<x)$: вероятность того, что величина *строго меньше* $x$. Для дискретной величины это сумма вероятностей всех значений левее $x$, и график — **лесенка**: ступенька высотой $p_i$ в каждой точке $x_i$.

> **Важно про `scipy`.** В `scipy` (и почти во всей англоязычной литературе) функция распределения определена как $P(X\le x)$. Для непрерывной величины разницы нет, потому что $P(X=x)=0$. Для дискретной разница есть ровно в точках $x_i$: по лекции $F(x_i)$ — значение *до* ступеньки, а `cdf(x_i)` в `scipy` — значение *после* неё. Поэтому для целочисленной величины $P(X<k)$ = `cdf(k - 1)`.

Возьмём ряд из задачи S08-03: $X$ — число тлей на листе.

In [ ]:
x_vals = np.array([0, 1, 2, 3, 4])
p_vals = np.array([0.30, 0.25, 0.20, 0.15, 0.10])
print("Сумма вероятностей:", p_vals.sum())

# Свой дискретный закон в scipy: rv_discrete(values=(значения, вероятности))
X = stats.rv_discrete(name="aphids", values=(x_vals, p_vals))

table = pd.DataFrame({
    "x": x_vals,
    "P(X = x)": X.pmf(x_vals),
    "F(x) = P(X < x), как на лекции": X.cdf(x_vals - 1),
    "scipy cdf(x) = P(X <= x)": X.cdf(x_vals),
}).set_index("x")
display(table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Многоугольник распределения
axes[0].vlines(x_vals, 0, p_vals, lw=6, alpha=0.4)
axes[0].plot(x_vals, p_vals, "o-", color="black")
axes[0].set_title("Многоугольник распределения")
axes[0].set_xlabel("x"); axes[0].set_ylabel("P(X = x)")

# Лесенка F(x) = P(X < x): на каждом полуинтервале (x_i; x_{i+1}] значение постоянно
edges = np.concatenate([[x_vals[0] - 1.5], x_vals, [x_vals[-1] + 1.5]])
levels = np.concatenate([[0], np.cumsum(p_vals)])
for a, b, F in zip(edges[:-1], edges[1:], levels):
    axes[1].hlines(F, a, b, lw=2.5)
axes[1].plot(x_vals, levels[:-1], "o", color="C0")                  # F(x_i) — нижнее значение (закрашенная точка)
axes[1].plot(x_vals, levels[1:], "o", mfc="white", color="C0")      # верхнее значение не достигается в x_i
axes[1].set_title("Функция распределения F(x) = P(X < x)")
axes[1].set_xlabel("x"); axes[1].set_ylabel("F(x)")
plt.tight_layout(); plt.show()

**Как читать лесенку.** Высота ступеньки в точке $x_i$ равна $P(X=x_i)$. Между значениями $F$ не меняется: вероятность «живёт» только в точках. Вероятность попасть в полуинтервал: $P(a\le X<b)=F(b)-F(a)$.

In [ ]:
def F_lecture(dist, x):
    # F(x) = P(X < x) для целочисленной величины
    return dist.cdf(np.ceil(x) - 1)

# P(1 <= X < 3) = F(3) - F(1)
print("P(1 <= X < 3) =", round(F_lecture(X, 3) - F_lecture(X, 1), 4))
print("Проверка суммой:", p_vals[1] + p_vals[2])

## 2. Проверяем задачи S08-01 – S08-03

**S08-01.** Три участка, вероятности заражения $0{,}1$, $0{,}2$, $0{,}3$. Ряд можно получить перебором всех $2^3=8$ исходов: так компьютер повторяет то, что мы делали у доски.

In [ ]:
from itertools import product

p_inf = [0.1, 0.2, 0.3]
law = {}
for outcome in product([0, 1], repeat=3):          # 1 — участок заражён
    prob = np.prod([p if o else 1 - p for o, p in zip(outcome, p_inf)])
    law[sum(outcome)] = law.get(sum(outcome), 0) + prob

X1 = stats.rv_discrete(values=(list(law), list(law.values())))
print("Ряд распределения:", {k: round(v, 4) for k, v in sorted(law.items())})
print("P(X >= 2) =", round(X1.sf(1), 4), " (sf(k) = P(X > k))")

# Симуляция: 100 000 сезонов
seasons = (rng.random((100_000, 3)) < p_inf).sum(axis=1)
print("Частоты в симуляции:", np.round(np.bincount(seasons, minlength=4) / len(seasons), 4))

**S08-02.** В ящике 10 саженцев, из них 3 с корневой гнилью; наудачу берут 3. Число больных среди взятых имеет **гипергеометрическое** распределение: `stats.hypergeom(M=всего, n=больных, N=берём)`.

In [ ]:
X2 = stats.hypergeom(M=10, n=3, N=3)
k = np.arange(4)
print(pd.DataFrame({"k": k, "P(X = k)": X2.pmf(k).round(4), "в долях 1/120": (X2.pmf(k) * 120).round(1)}).set_index("k"))
print("P(X <= 1) =", round(X2.cdf(1), 4))

**S08-03** уже разобрана в разделе 1.

## 3. Непрерывная случайная величина

У непрерывной величины $P(X=a)=0$ для любого $a$, поэтому ряд не работает. Вместо него — **плотность** $f(x)=F'(x)$:

$$
P(a<X<b)=\int_a^b f(x)\,dx = F(b)-F(a),\qquad F(x)=\int_{-\infty}^x f(t)\,dt,\qquad \int_{-\infty}^{+\infty} f(x)\,dx=1.
$$

Вероятность — это **площадь под кривой плотности**, а значение $F(x)$ — площадь левее $x$.

Задача S08-04: время прорастания семени $X$ (сутки) имеет плотность $f(x)=c\,x(4-x)$ на $[0;4]$ и $0$ вне отрезка.

In [ ]:
f_raw = lambda x: x * (4 - x)
area, _ = integrate.quad(f_raw, 0, 4)
c = 1 / area
print(f"Интеграл x(4-x) от 0 до 4 = {area:.4f}  ->  c = {c:.5f} = 3/32 = {3/32:.5f}")

def f(x):
    x = np.asarray(x, dtype=float)
    return np.where((x >= 0) & (x <= 4), c * x * (4 - x), 0.0)

def F(x):
    x = np.clip(np.asarray(x, dtype=float), 0, 4)
    return 3 * x**2 / 16 - x**3 / 32

print("P(1 < X < 2) через интеграл:", round(integrate.quad(f, 1, 2)[0], 5))
print("P(1 < X < 2) через F:        ", round(F(2) - F(1), 5), "= 11/32 =", 11 / 32)
print("P(X > 3) = 1 - F(3) =", round(1 - F(3), 5), "= 5/32")

In [ ]:
xs = np.linspace(-0.5, 4.5, 500)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(xs, f(xs), color="C0", lw=2)
mask = (xs > 1) & (xs < 2)
axes[0].fill_between(xs[mask], f(xs[mask]), alpha=0.35, label=f"P(1<X<2) = площадь = {F(2) - F(1):.4f}")
axes[0].set_title("Плотность f(x)"); axes[0].set_xlabel("сутки"); axes[0].legend()

axes[1].plot(xs, F(xs), color="C1", lw=2)
for x0 in (1, 2):
    axes[1].plot([x0, x0], [0, F(x0)], "k:"); axes[1].plot([-0.5, x0], [F(x0), F(x0)], "k:")
axes[1].annotate("", xy=(2.15, F(2)), xytext=(2.15, F(1)), arrowprops=dict(arrowstyle="<->", color="C0"))
axes[1].text(2.25, (F(1) + F(2)) / 2, "F(2) - F(1)", color="C0")
axes[1].set_title("Функция распределения F(x)"); axes[1].set_xlabel("сутки")
plt.tight_layout(); plt.show()

**Связь двух графиков.** Площадь закрашенной полосы слева равна длине стрелки справа. Там, где плотность выше, $F(x)$ растёт круче: $f$ — это скорость роста $F$.

### Плотность — не вероятность

Плотность может быть больше единицы. Для равномерного закона на $[0;\,0{,}1]$ плотность равна $10$, а вероятности всё равно не больше 1, потому что ширина отрезка мала.

In [ ]:
U = stats.uniform(loc=0, scale=0.1)        # равномерный на [loc; loc + scale]
print("f(0,05) =", U.pdf(0.05), "   P(0 < X < 0,1) =", U.cdf(0.1) - U.cdf(0))

# А вероятность попасть в одну точку равна нулю: сужаем интервал вокруг 0,05
for h in [0.01, 0.001, 0.0001]:
    print(f"P(0,05 - {h} < X < 0,05 + {h}) = {U.cdf(0.05 + h) - U.cdf(0.05 - h):.5f}")

## 4. Проверяем задачи S08-05 и S08-06

**S08-05.** Срок службы насоса $T$ (годы): $F(t)=1-e^{-0{,}5t}$ при $t\ge0$. Это **показательный** (экспоненциальный) закон с $\lambda=0{,}5$.

> В `scipy` показательный закон задаётся не через $\lambda$, а через `scale` $=1/\lambda$: `stats.expon(scale=2)`. Это самая частая ошибка при переходе от учебника к коду.

In [ ]:
T = stats.expon(scale=1 / 0.5)
print("f(t) = 0,5 e^{-0,5t}; f(1) =", round(T.pdf(1), 4), "  проверка:", round(0.5 * math.exp(-0.5), 4))
print("P(T > 2)     =", round(T.sf(2), 4), "  = e^{-1}")
print("P(1 < T < 3) =", round(T.cdf(3) - T.cdf(1), 4))
print("Половина насосов откажет к t =", round(T.ppf(0.5), 4), "года  = 2 ln 2")

**S08-06.** Дождевальная машина проходит над точкой поля каждые 30 минут, агроном приходит в случайный момент. Время ожидания $X$ равномерно на $[0;30]$.

In [ ]:
W = stats.uniform(loc=0, scale=30)
print("f(x) =", round(W.pdf(10), 4), "= 1/30 на [0; 30]")
print("P(X < 10)       =", round(W.cdf(10), 4))
print("P(5 < X < 20)   =", round(W.cdf(20) - W.cdf(5), 4))
print("P(X = 15) = 0, а f(15) =", round(W.pdf(15), 4))

## 5. Гид по `scipy.stats`

Любое распределение в `scipy` — объект с одинаковым набором методов. Сначала «замораживаем» параметры, потом спрашиваем.

| Метод | Что считает | Формула |
|---|---|---|
| `pmf(k)` | вероятность значения (только дискретные) | $P(X=k)$ |
| `pdf(x)` | плотность (только непрерывные) | $f(x)$ |
| `cdf(x)` | функция распределения в соглашении `scipy` | $P(X\le x)$ |
| `sf(x)` | «хвост», функция выживания | $1-$`cdf(x)` $=P(X>x)$ |
| `ppf(q)` | квантиль: обратная к `cdf` | такое $x$, что `cdf(x)` $=q$ |
| `isf(q)` | обратная к `sf` | такое $x$, что $P(X>x)=q$ |
| `rvs(size, random_state)` | случайная выборка | симуляция |

| Закон | Учебник | `scipy` |
|---|---|---|
| биномиальный | $n$, $p$ | `stats.binom(n, p)` |
| Пуассона | $\lambda$ | `stats.poisson(mu=λ)` |
| гипергеометрический | всего $N$, «особых» $M$, берём $n$ | `stats.hypergeom(M=N, n=M, N=n)` (буквы перепутаны!) |
| свой ряд | таблица | `stats.rv_discrete(values=(x, p))` |
| равномерный на $[a;b]$ | $a$, $b$ | `stats.uniform(loc=a, scale=b-a)` |
| показательный | $\lambda$ | `stats.expon(scale=1/λ)` |

Ниже — одна и та же величина глазами всех методов.

In [ ]:
T = stats.expon(scale=2)            # срок службы насоса из S08-05
print("pdf(1) =", T.pdf(1))
print("cdf(1) =", T.cdf(1), " sf(1) =", T.sf(1), " сумма =", T.cdf(1) + T.sf(1))
print("ppf(cdf(1)) =", T.ppf(T.cdf(1)), " — квантиль возвращает нас в точку 1")
print("Отрезок, куда T попадает с вероятностью 0,9 (по 5 % отрезаем с краёв):", np.round(T.ppf([0.05, 0.95]), 3))
print("Пять случайных насосов:", np.round(T.rvs(size=5, random_state=rng), 2))

**Свой непрерывный закон.** Если нужного закона в `scipy` нет, его можно описать, задав только плотность (или только $F$). Остальные методы `scipy` досчитает сам, численно. Возьмём плотность из S08-04.

In [ ]:
class Germination(stats.rv_continuous):
    # Время прорастания: f(x) = 3/32 · x(4 - x) на [0; 4]
    def _pdf(self, x):
        return 3 / 32 * x * (4 - x)

G = Germination(a=0, b=4, name="germination")
print("P(1 < X < 2) =", round(G.cdf(2) - G.cdf(1), 5))
print("К какому дню прорастёт половина семян (T50):", round(G.ppf(0.5), 4))
print("К какому дню прорастёт 90 % семян (T90):    ", round(G.ppf(0.9), 4))

**Вопрос:** почему T50 получилось ровно 2? Посмотрите на график плотности в разделе 3.

## 6. Из данных в распределение

На практике формулу $F(x)$ нам никто не даёт: у нас есть только наблюдения. Как по ним восстановить распределение?

- **Эмпирическая функция распределения** $F_n(x)$ — доля наблюдений, меньших $x$. Это та же лесенка, что в разделе 1, только каждая ступенька высотой $1/n$.
- **Гистограмма плотности** — столбики высотой «доля наблюдений в столбике / ширина столбика», так что общая площадь равна 1. Это приближение к $f(x)$.

Посадим $n$ семян и запишем время прорастания каждого.

In [ ]:
def ecdf(sample, x):
    # Эмпирическая функция распределения: доля наблюдений, строго меньших x
    sample = np.sort(sample)
    return np.searchsorted(sample, x, side="left") / len(sample)

xs = np.linspace(0, 4, 400)
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for j, n in enumerate([20, 200, 5000]):
    sample = G.rvs(size=n, random_state=rng)
    axes[0, j].hist(sample, bins=16, range=(0, 4), density=True, alpha=0.5, label="гистограмма")
    axes[0, j].plot(xs, G.pdf(xs), "k", lw=2, label="плотность f(x)")
    axes[0, j].set_title(f"n = {n}")
    axes[1, j].step(xs, ecdf(sample, xs), where="post", label="эмпирическая F_n(x)")
    axes[1, j].plot(xs, G.cdf(xs), "k--", lw=1.5, label="F(x)")
    gap = np.abs(ecdf(sample, xs) - G.cdf(xs)).max()
    axes[1, j].set_title(f"макс. расхождение = {gap:.3f}")
axes[0, 0].legend(); axes[1, 0].legend()
plt.tight_layout(); plt.show()

**Что видно.** С ростом $n$ эмпирическая функция прилипает к настоящей $F(x)$: максимальное расхождение убывает примерно как $1/\sqrt{n}$ (теорема Гливенко — Кантелли). Гистограмма при малом $n$ «скачет» сильнее, чем лесенка: оценить $F$ по данным проще, чем $f$.

### Ширина столбика решает

Одни и те же 300 наблюдений с разным числом столбиков:

In [ ]:
sample = G.rvs(size=300, random_state=rng)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, bins in zip(axes, [3, 15, 120]):
    ax.hist(sample, bins=bins, range=(0, 4), density=True, alpha=0.5)
    ax.plot(xs, G.pdf(xs), "k", lw=2)
    ax.set_title(f"{bins} столбиков")
plt.tight_layout(); plt.show()

**Вопрос:** что «врёт» на левой картинке, а что на правой? Почему у эмпирической функции распределения такой проблемы нет?

## 7. Оцениваем параметр по данным

Пусть срок службы насосов показательный, но $\lambda$ неизвестна. Мы наблюдали 50 насосов. Как оценить $\lambda$, **используя только функцию распределения**?

Идея: $P(T>t)=e^{-\lambda t}$. Долю насосов, проработавших дольше $t$, мы видим в данных, значит

$$
\hat\lambda=-\frac{\ln\big(\text{доля насосов с } T>t\big)}{t}.
$$

In [ ]:
LAMBDA_TRUE = 0.5
pumps = stats.expon(scale=1 / LAMBDA_TRUE).rvs(size=50, random_state=rng)

t0 = 2.0
share_alive = np.mean(pumps > t0)
lam_hat = -math.log(share_alive) / t0
print(f"Проработали дольше {t0} лет: {share_alive:.2f}  ->  оценка λ = {lam_hat:.3f} (истинное {LAMBDA_TRUE})")

xs_t = np.linspace(0, 10, 300)
plt.step(xs_t, ecdf(pumps, xs_t), where="post", label="данные: 50 насосов")
plt.plot(xs_t, 1 - np.exp(-lam_hat * xs_t), label=f"F(t) с оценкой λ = {lam_hat:.2f}")
plt.plot(xs_t, 1 - np.exp(-LAMBDA_TRUE * xs_t), "k--", label="истинная F(t)")
plt.axvline(t0, color="gray", ls=":")
plt.xlabel("годы"); plt.legend(); plt.show()

**Насколько точна оценка?** Повторим «наблюдение за 50 насосами» 2000 раз и посмотрим, как разбросаны оценки. Затем то же для 500 насосов.

In [ ]:
def estimate_lambda(n, t0=2.0, reps=2000):
    data = stats.expon(scale=1 / LAMBDA_TRUE).rvs(size=(reps, n), random_state=rng)
    share = np.mean(data > t0, axis=1)
    share = np.clip(share, 1 / n, 1 - 1 / n)          # защита от ln(0)
    return -np.log(share) / t0

fig, ax = plt.subplots()
for n in [50, 500]:
    est = estimate_lambda(n)
    lo, hi = np.quantile(est, [0.05, 0.95])
    ax.hist(est, bins=50, density=True, alpha=0.5, label=f"n = {n}: 90 % оценок в [{lo:.2f}; {hi:.2f}]")
ax.axvline(LAMBDA_TRUE, color="black", lw=2, label="истинное λ")
ax.set_xlabel("оценка λ"); ax.legend(); plt.show()

**Вывод.** Оценка сама по себе случайная величина со своим распределением. При $n=50$ она промахивается заметно, при $n=500$ отрезок, куда попадают 90 % оценок, сужается примерно в $\sqrt{10}\approx3$ раза.

**Ещё два способа оценить параметр по $F$:**

- *Равномерный закон на $[0;b]$:* $b$ неизвестно. Самое большое наблюдение всегда немного меньше $b$, поэтому берут оценку $\max\cdot\frac{n+1}{n}$.
- *Квантиль:* T50 (время, к которому проросла половина семян) оценивается медианой выборки, то есть точкой, где эмпирическая функция переходит через $0{,}5$.

In [ ]:
# Равномерный закон: интервал прохода дождевальной машины b неизвестен (истинное 30 минут)
waits = stats.uniform(0, 30).rvs(size=20, random_state=rng)
print("max =", round(waits.max(), 2), " -> оценка b =", round(waits.max() * 21 / 20, 2))

# T50 по выборке из 40 семян против истинного T50 = 2
germ = G.rvs(size=40, random_state=rng)
print("T50 по выборке:", round(np.median(germ), 3), " истинное:", round(G.ppf(0.5), 3))

## 8. Как компьютер получает величину с нужной $F(x)$

Генератор умеет только одно: выдавать $U$, равномерное на $[0;1]$. Чтобы получить величину с функцией распределения $F$, достаточно взять

$$
X=F^{-1}(U).
$$

Почему: $P(X<x)=P(F^{-1}(U)<x)=P(U<F(x))=F(x)$. Для показательного закона $F^{-1}(u)=-\ln(1-u)/\lambda$.

In [ ]:
u = rng.random(20_000)
lam = 0.5
x_inv = -np.log(1 - u) / lam            # обратная функция к F(t) = 1 - e^{-λt}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(u, bins=40, density=True, alpha=0.5); axes[0].set_title("U: равномерные на [0; 1]")
axes[1].hist(x_inv, bins=60, range=(0, 12), density=True, alpha=0.5, label="F⁻¹(U)")
tt = np.linspace(0, 12, 300)
axes[1].plot(tt, stats.expon(scale=1 / lam).pdf(tt), "k", lw=2, label="плотность показательного закона")
axes[1].set_title("X = F⁻¹(U)"); axes[1].legend()
plt.tight_layout(); plt.show()

**Вопрос:** для дискретной величины из раздела 1 (тли на листе) как превратить $U$ в число тлей? Подсказка: разбейте отрезок $[0;1]$ на кусочки длиной $p_i$.

In [ ]:
# Ответ на вопрос: кусочки [0; 0,3), [0,3; 0,55), ... -> номер кусочка = число тлей
cut = np.cumsum(p_vals)
aphids = np.searchsorted(cut, rng.random(100_000), side="right")
print("Частоты:", np.round(np.bincount(aphids, minlength=5) / 100_000, 3), " теория:", p_vals)

## 9. Практика: засуха и обеспеченность осадков

В агрометеорологии и гидрологии говорят об **обеспеченности** осадков: вероятности того, что за сезон выпадет *не меньше* данного количества. Это $1-F(x)$. «Осадки 75 %-й обеспеченности» — уровень, который превышается в 3 годах из 4, по нему планируют орошение. Вероятность засухи — $F(\text{порог})$.

Ниже 40 «лет наблюдений» за осадками вегетационного периода (мм). Данные сгенерированы, но похожи на реальные для средней полосы.

In [ ]:
rain = np.round(stats.gamma(a=9, scale=32).rvs(size=40, random_state=np.random.default_rng(7)))
print("Осадки по годам, мм:", rain.astype(int))

DROUGHT = 200
print(f"Доля лет с осадками меньше {DROUGHT} мм (оценка F({DROUGHT})): {np.mean(rain < DROUGHT):.3f}")
print("Осадки 75 %-й обеспеченности (превышаются в 75 % лет):", np.quantile(rain, 0.25), "мм")

xr = np.linspace(rain.min() - 20, rain.max() + 20, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].step(xr, ecdf(rain, xr), where="post")
axes[0].axvline(DROUGHT, color="red", ls="--", label="порог засухи")
axes[0].set_title("Эмпирическая F(x)"); axes[0].set_xlabel("мм"); axes[0].legend()
axes[1].step(xr, 1 - ecdf(rain, xr), where="post", color="C2")
axes[1].axhline(0.75, color="gray", ls=":")
axes[1].set_title("Кривая обеспеченности 1 − F(x)"); axes[1].set_xlabel("мм")
plt.tight_layout(); plt.show()

**Вопросы:**

1. В наших 40 годах нет ни одного года с осадками меньше 100 мм. Значит ли это, что вероятность такой засухи равна нулю?
2. Как изменилась бы кривая обеспеченности, если бы климат стал суше и все значения сдвинулись влево на 30 мм?

---

## Итоги

1. Дискретная величина задаётся рядом распределения, непрерывная — плотностью. Функция распределения $F(x)$ есть у обеих: у дискретной это лесенка, у непрерывной — гладкая кривая.
2. Вероятность попасть в интервал равна $F(b)-F(a)$; для непрерывной величины это ещё и площадь под плотностью. Плотность — не вероятность и может быть больше 1.
3. По данным $F(x)$ восстанавливается эмпирической функцией распределения, $f(x)$ — гистограммой; параметры закона можно оценить, приравняв теоретическую $F$ к наблюдаемой доле.
4. В `scipy`: `pmf`/`pdf`, `cdf` (это $P(X\le x)$!), `sf`, `ppf`, `rvs`; у показательного закона `scale = 1/λ`.